# B2-020-language-transformers — Practice p16 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** masked-language-modeling, nlp-pretraining-objectives

*45 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:C7-pytorch`, `aio-ml:C10-cnn-transfer`, `aio-ml:C8-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C10-cnn-transfer](../../../../aio-ml/units/17-C10-cnn-transfer/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

Take the sequence `[red, blue, red]` and select the middle `blue` as the target. If the input remains `[red, blue, red]`, a position-wise copy head can put arbitrarily large probability on the visible middle ID, driving selected-token loss toward zero without learning context. Replace the selected input by `<mask>` to get `[red, <mask>, red]`; now copying cannot reveal `blue`. Labels are `[-100, blue, -100]`, so only the hidden position contributes to loss and unselected copy predictions receive no credit.

In [ ]:
import torch
visible = torch.tensor([4, 5, 4])
selected = torch.tensor([False, True, False])
corrupted = visible.clone(); corrupted[selected] = 1
labels = torch.full_like(visible, -100); labels[selected] = visible[selected]

### Answer check

In [ ]:
# Checks: the selected token's corruption and label
assert visible[selected].item() == 5
assert corrupted[selected].item() == 1
assert labels.tolist() == [-100, 5, -100]
assert not torch.equal(corrupted[selected], labels[selected])